1. Загрузите набор данных lenta-ru-news с помощью библиотеки Corus или любым другим способом для задачи классификации текстов по топикам (пригодятся атрибуты title, text, topic)

In [ ]:
%%capture
!pip install corus razdel pymorphy3

In [ ]:
!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

--2026-04-12 19:35:45--  https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/87156914/0b363e00-0126-11e9-9e3c-e8c235463bd6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-12T20%3A20%3A55Z&rscd=attachment%3B+filename%3Dlenta-ru-news.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-12T19%3A20%3A11Z&ske=2026-04-12T20%3A20%3A55Z&sks=b&skv=2018-11-09&sig=YQ2UJOmEsvZbPSQEBqnb27vCFEhx2XboDMkj4Qtt%2F%2FU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NjAyNjE0NSwibmJmIjoxNzc2MDIyNTQ1LCJwYXRoIjoicmVsZWFzZWFzc2

In [ ]:
from corus import load_lenta

path = 'lenta-ru-news.csv.gz'
c = load_lenta(path)
next(records)

LentaRecord(
    url='https://lenta.ru/news/2018/12/14/cancer/',
    title='Названы регионы России с\xa0самой высокой смертностью от\xa0рака',
    text='Вице-премьер по социальным вопросам Татьяна Голикова рассказала, в каких регионах России зафиксирована наиболее высокая смертность от рака, сообщает РИА Новости. По словам Голиковой, чаще всего онкологические заболевания становились причиной смерти в Псковской, Тверской, Тульской и Орловской областях, а также в Севастополе. Вице-премьер напомнила, что главные факторы смертности в России — рак и болезни системы кровообращения. В начале года стало известно, что смертность от онкологических заболеваний среди россиян снизилась впервые за три года. По данным Росстата, в 2017 году от рака умерли 289 тысяч человек. Это на 3,5 процента меньше, чем годом ранее.',
    topic='Россия',
    tags='Общество',
    date=None
)

In [ ]:
dataset = [next(records).text for i in range(100000)]
len(dataset)

100000

In [ ]:
dataset[0]

'Поклонники фантастической киносаги Джорджа Лукаса «Звездные войны» собрались в нескольких городах США с игрушечными световыми мечами, чтобы почтить память скончавшейся актрисы Кэрри Фишер, сыгравшей во франшизе принцессу Лею. Об этом в четверг, 29 декабря, сообщает Mashable.  Фанаты устроили массовые флешмобы в Остине, штат Техас, а также в Анахайме, штат Калифорния. Многие участники мероприятия пришли в костюмах персонажей из вселенной «Звездных войн».  Ранее 29 декабря поклонники Фишер установили актрисе самодельную звезду на голливудской Аллее славы. Официально артистке эта награда не присуждалась. Фишер умерла в больнице во вторник, 27 декабря, в возрасте 60 лет от остановки сердца. Ее мать, актриса Дебби Рейнольдс, скончалась на следующий день после смерти дочери. 24 декабря сообщалось, что Фишер была госпитализирована в критическом состоянии из-за сердечного приступа, произошедшего во время перелета из Лондона в Лос-Анджелес. Врачи доставили актрису в больницу после приземления.

Подготовьте данные к обучению: - 2 балла  



*   Предобработайте данные: реализуйте оптимальную, на ваш взгляд, предобработку текстов (нормализация, очистка, стемминг/лемматизация и т.п.) и таргета.

*   hint: для ускорения обработки и обучения можно ограничиться не всем датасетом, а его репрезентативной частью, например, размера 100_000.

*   Кратко опишите пайплайн, на котором остановились, и почему.
*   Разделите датасет на обучающую, валидационную и тестовую выборки со стратификацией в пропорции 60/20/20. В качестве целевой переменной используйте атрибут topic

In [ ]:
from bs4 import BeautifulSoup
import requests

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from string import punctuation

# Скачать стоп-слова (один раз)
nltk.download('stopwords')
russian_stopwords = stopwords.words('russian')
# Добавим специфичные для новостей стоп-слова
extra_stopwords = ['это', 'так', 'вот', 'сказал', 'сообщил', 'передает', 'пояснил']
russian_stopwords.extend(extra_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def advanced_clean_text(text, remove_stopwords=False, lemmatize=False):
    """Продвинутая очистка с опциями"""

    # Базовая очистка
    text = html.unescape(text)
    text = text.replace('\xa0', ' ')
    text = BeautifulSoup(text, 'html.parser').get_text()

    # Удаление URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Удаление email
    text = re.sub(r'\S+@\S+', '', text)

    # Удаление чисел (опционально)
    # text = re.sub(r'\d+', '', text)

    # Удаление пунктуации
    text = re.sub(r'[{}]'.format(punctuation), ' ', text)

    # Приведение к нижнему регистру
    text = text.lower()

    # Удаление стоп-слов
    if remove_stopwords:
        words = text.split()
        words = [word for word in words if word not in russian_stopwords]
        text = ' '.join(words)

    # Лемматизация (медленно, но качественно)
    if lemmatize:
        from pymystem3 import Mystem
        m = Mystem()
        lemmas = m.lemmatize(text)
        text = ''.join(lemmas)

    # Нормализация пробелов
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
for data in soup(['style', 'script']):
  data.decompose()
' '.join(soup.stripped_strings)

In [ ]:
# Проверьте, есть ли HTML теги в первом документе
print(dataset[0][:500])  # Посмотрите первые 500 символов

Поклонники фантастической киносаги Джорджа Лукаса «Звездные войны» собрались в нескольких городах США с игрушечными световыми мечами, чтобы почтить память скончавшейся актрисы Кэрри Фишер, сыгравшей во франшизе принцессу Лею. Об этом в четверг, 29 декабря, сообщает Mashable.  Фанаты устроили массовые флешмобы в Остине, штат Техас, а также в Анахайме, штат Калифорния. Многие участники мероприятия пришли в костюмах персонажей из вселенной «Звездных войн».  Ранее 29 декабря поклонники Фишер установ


In [ ]:
for record in dataset:
    if hasattr(record, 'title'):
        clean_title = record.title.replace('\xa0', ' ')
        # или record.title = BeautifulSoup(record.title, 'html.parser').get_text()

AttributeError: 'builtin_function_or_method' object has no attribute 'replace'

In [ ]:
import re
import html
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import nltk

# Скачиваем стоп-слова (один раз)
nltk.download('stopwords')

# Загрузка данных
from corus import load_lenta

path = 'lenta-ru-news.csv.gz'
c = load_lenta(path)

# Загружаем 100000 записей с текстом и темой
dataset = []
for i, record in enumerate(c):
    if i >= 1000:
        break
    # Пропускаем пустые темы
    topic = record.topic
    if not topic or str(topic).strip() == '':  # отфильтровываем
        continue
    dataset.append({
        'text': record.text,
        'topic': record.topic
    })


import re
import html
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import nltk

# Скачиваем стоп-слова (один раз)
nltk.download('stopwords')

def full_preprocessing(text, normalize_method='stemming'):
    """
    Полная предобработка текста: очистка + нормализация

    Parameters:
    - text: исходный текст
    - normalize_method: 'stemming' или 'lemmatization'
    """
    # ========== 1. ОЧИСТКА ==========
    # HTML сущности и декодирование
    text = html.unescape(text)

    # Замена неразрывных пробелов
    text = text.replace('\xa0', ' ').replace('\u2009', ' ')

    # Удаление HTML тегов
    text = BeautifulSoup(text, 'html.parser').get_text()

    # Удаление URL и email
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # Приведение к нижнему регистру
    text = text.lower()

    # Удаление цифр и пунктуации (оставляем только буквы и пробелы)
    text = re.sub(r'[^а-яё\s]', ' ', text, flags=re.IGNORECASE)

    # ========== 2. ТОКЕНИЗАЦИЯ (простая) ==========
    words = text.split()

    # Удаление стоп-слов и коротких слов
    russian_stopwords = stopwords.words('russian')
    extra_stopwords = ['это', 'так', 'вот', 'сказал', 'сообщил', 'передает',
                       'пояснил', 'отметил', 'добавил', 'рассказал']
    all_stopwords = set(russian_stopwords + extra_stopwords)

    words = [w for w in words if w not in all_stopwords and len(w) > 2]

    # ========== 3. НОРМАЛИЗАЦИЯ ==========
    if normalize_method == 'stemming':
        # Стемминг (быстрее, но грубее)
        stemmer = SnowballStemmer('russian')
        words = [stemmer.stem(w) for w in words]
    elif normalize_method == 'lemmatization':
        # Лемматизация (точнее, но медленнее)
        from pymorphy3 import MorphAnalyzer
        morph = MorphAnalyzer()
        words = [morph.parse(w)[0].normal_form for w in words]

    # Собираем обратно
    text = ' '.join(words)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# Пример работы
example = "В России очень сильно выросла смертность от рака легких"
print(f"Оригинал: {example}")
print(f"Стемминг: {full_preprocessing(example, 'stemming')}")
print(f"Лемматизация: {full_preprocessing(example, 'lemmatization')}")



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Оригинал: В России очень сильно выросла смертность от рака легких
Стемминг: росс очен сильн выросл смертност рак легк
Лемматизация: россия очень сильно вырасти смертность рак лёгкий


In [ ]:
for item in dataset:
    clean_text = full_preprocessing(item['text'], 'lemmatization')
    clean_texts.append(clean_text)
    topics.append(item['topic'])

In [ ]:
topics[:10]

['Россия',
 'Спорт',
 'Путешествия',
 'Мир',
 'Мир',
 'Бывший СССР',
 'Интернет и СМИ',
 'Мир',
 'Мир',
 'Силовые структуры']

In [ ]:
# Предобработка текста
clean_text = preprocess_text(record.text)

In [ ]:
# Проверяем распределение тем
from collections import Counter
topic_counts = Counter(topics)
print("\nРаспределение тем:")
for topic, count in topic_counts.most_common(1000):
    print(f"  {topic}: {count} ({count/len(topics)*100:.1f}%)")


Распределение тем:
  Россия: 15151 (15.2%)
  Мир: 14421 (14.4%)
  Спорт: 10045 (10.0%)
  Экономика: 7682 (7.7%)
  Интернет и СМИ: 6935 (6.9%)
  Силовые структуры: 6925 (6.9%)
  Бывший СССР: 6810 (6.8%)
  Культура: 6578 (6.6%)
  Наука и техника: 5645 (5.6%)
  Из жизни: 4903 (4.9%)
  Ценности: 4480 (4.5%)
  Дом: 3408 (3.4%)
  Путешествия: 3223 (3.2%)
  Бизнес: 1993 (2.0%)
  69-я параллель: 815 (0.8%)
  Крым: 661 (0.7%)
  Культпросвет : 307 (0.3%)
  Оружие: 1 (0.0%)


In [ ]:
#Было 17 текстов с пустым топиком, я их отфильтровал при загрузке
#Выбрал стемминг только из-за скорости, попробовав на 1000 текстов

In [9]:
#!pip install corus
#!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz
import re
import html
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from collections import Counter
from corus import load_lenta
import nltk
import warnings
warnings.filterwarnings('ignore')

1. ЗАГРУЗКА ДАННЫХ

In [12]:
path = 'lenta-ru-news.csv.gz'
c = load_lenta(path)
raw_data = []

#Загружаю чуть больше, чтобы потом была возможность убрать "лишнее"
for i, record in enumerate(c):
    if i >= 110000:
        break
    raw_data.append({
        'title': getattr(record, 'title', ''),
        'text': record.text,
        'topic': record.topic
    })


len(raw_data)

110000

2. ПРЕДОБРАБОТКА ДАННЫХ

In [17]:
# Скачиваем стоп-слова
nltk.download('stopwords', quiet=True)

# Доп. стоп-слова из тематики новостей
russian_stopwords = set(stopwords.words('russian'))
extra_stopwords = {
    'это', 'так', 'вот', 'сказал', 'сообщил', 'передает', 'пояснил',
    'отметил', 'добавил', 'рассказал', 'стал', 'стало', 'является',
    'находится', 'можно', 'нужно', 'также', 'например', 'ря', 'год',
    'года', 'лет', 'дня', 'дней', 'месяца', 'месяцев'
}
all_stopwords = russian_stopwords | extra_stopwords

In [18]:
# Инициализируем стеммер
stemmer = SnowballStemmer('russian')

def preprocess_text(text):

    if not text or not isinstance(text, str):
        return ""

    # Очистка HTML и сущностей
    text = html.unescape(text)
    text = text.replace('\xa0', ' ').replace('\u2009', ' ')
    text = BeautifulSoup(text, 'html.parser').get_text()

    # Удаление URL, email, цифр
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)

    # Приведение к нижнему регистру
    text = text.lower()

    # Удаление пунктуации и спецсимволов
    text = re.sub(r'[^а-яё\s]', ' ', text)

    # Токенизация и удаление стоп-слов
    words = text.split()
    words = [w for w in words if w not in all_stopwords and len(w) > 2]

    # Стемминг
    #Выбрал стемминг только из-за скорости, на 1000 текстов разница ~20 раз
    words = [stemmer.stem(w) for w in words]

    return ' '.join(words)

#Были тексты с пустым топиком, лучше убрать перед обучением
def preprocess_topic(topic):
    if topic is None or topic == '':
        return None
    return topic

In [36]:
processed_data = []

for item in raw_data:
    # Обрабатываем текст (объединяем title и text для большей информативности)
    full_text = f"{item['title']} {item['text']}"
    clean_text = preprocess_text(full_text)

    # Обрабатываем тему
    clean_topic = preprocess_topic(item['topic'])

    # Пропускаем пустые тексты и пустые темы
    if clean_text and len(clean_text.split()) > 10 and clean_topic is not None:
        processed_data.append({
            'text': clean_text,
            'topic': clean_topic
        })

In [42]:
# Анализ распределения тем
topics_dist = Counter([d['topic'] for d in processed_data])
print(f"\nРаспределение тем (топ-15):")
for topic, count in topics_dist.most_common(15):
    print(f"  Тема {topic}: {count} ({count/len(processed_data)*100:.2f}%)")

# Фильтруем редкие темы
min_samples_per_topic = 5
valid_topics = {topic for topic, count in topics_dist.items() if count >= min_samples_per_topic}
filtered_data = [d for d in processed_data if d['topic'] in valid_topics]
print(f"После фильтрации редких тем: {len(filtered_data)} новостей")

# Подготовка массивов
X = [d['text'] for d in filtered_data]
y = [d['topic'] for d in filtered_data]


Распределение тем (топ-15):
  Тема Россия: 16744 (15.22%)
  Тема Мир: 15958 (14.51%)
  Тема Спорт: 10986 (9.99%)
  Тема Экономика: 8271 (7.52%)
  Тема Силовые структуры: 8014 (7.29%)
  Тема Интернет и СМИ: 7543 (6.86%)
  Тема Бывший СССР: 7517 (6.83%)
  Тема Культура: 7228 (6.57%)
  Тема Наука и техника: 6102 (5.55%)
  Тема Из жизни: 5264 (4.79%)
  Тема Ценности: 4838 (4.40%)
  Тема Путешествия: 3614 (3.29%)
  Тема Дом: 3590 (3.26%)
  Тема Бизнес: 2406 (2.19%)
  Тема 69-я параллель: 885 (0.80%)
После фильтрации редких тем: 109979 новостей


3. РАЗДЕЛЕНИЕ НА ВЫБОРКИ

In [46]:
# Подготовка массивов
X = [d['text'] for d in filtered_data]
y = [d['topic'] for d in filtered_data]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(f"Обучающая выборка: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Валидационная выборка: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Тестовая выборка: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

# Проверка стратификации
print(f"\nУникальных тем в train: {len(set(y_train))}")
print(f"Уникальных тем в val: {len(set(y_val))}")
print(f"Уникальных тем в test: {len(set(y_test))}")


Обучающая выборка: 65987 (60.0%)
Валидационная выборка: 21996 (20.0%)
Тестовая выборка: 21996 (20.0%)

Уникальных тем в train: 18
Уникальных тем в val: 18
Уникальных тем в test: 18


4. ОБУЧЕНИЕ С COUNTVECTORIZER

In [49]:
count_vectorizer = CountVectorizer(
    max_features=15000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

X_train_count = count_vectorizer.fit_transform(X_train)
X_val_count = count_vectorizer.transform(X_val)

lr_count = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

lr_count.fit(X_train_count, y_train)
y_val_pred_count = lr_count.predict(X_val_count)

print(f"CountVectorizer + LogisticRegression")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred_count):.4f}")

CountVectorizer + LogisticRegression
Accuracy: 0.8416


5. ОБУЧЕНИЕ С TFIDFVECTORIZER

In [48]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

lr_tfidf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

lr_tfidf.fit(X_train_tfidf, y_train)
y_val_pred_tfidf = lr_tfidf.predict(X_val_tfidf)

print(f"TfidfVectorizer + LogisticRegression")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred_tfidf):.4f}")

TfidfVectorizer + LogisticRegression
Accuracy: 0.8357


In [50]:
#С базовыми гиперпараметрами себя лучше показал CountVectorizer, поэтому буду подбирать гиперпараметры к нему (как будто бы незначимо)

6. ОПТИМИЗАЦИЯ ГИПЕРПАРАМЕТРОВ

In [52]:
# GridSearch для оптимизации гиперпараметров
param_grid = {
    'C': [0.01, 0.1, 1.0],
    'solver': ['lbfgs', 'liblinear'],
    'penalty': ['l2']
}

lr_grid = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

grid_search = GridSearchCV(
    lr_grid,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_count, y_train)

print(f"\nЛучшие параметры: {grid_search.best_params_}")
print(f"Лучшая кросс-валидационная точность: {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 6 candidates, totalling 18 fits

Лучшие параметры: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Лучшая кросс-валидационная точность: 0.8419


In [53]:
# Оценка на валидационной выборке
best_model = grid_search.best_estimator_
y_val_pred_optimized = best_model.predict(X_val_count)
val_accuracy_optimized = accuracy_score(y_val, y_val_pred_optimized)
print(f"Accuracy (оптимизированная): {val_accuracy_optimized:.4f}")

Accuracy (оптимизированная): 0.8542


In [57]:
# Преобразуем тестовую выборку
X_test_best = count_vectorizer.transform(X_test)

# Предсказания
y_test_pred = best_model.predict(X_test_best)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Детальный отчет
print("\nClassification Report на тестовой выборке:")
print(classification_report(y_test, y_test_pred))


Classification Report на тестовой выборке:
                   precision    recall  f1-score   support

   69-я параллель       0.86      0.88      0.87       177
           Бизнес       0.59      0.58      0.59       481
      Бывший СССР       0.87      0.88      0.88      1503
              Дом       0.87      0.85      0.86       718
         Из жизни       0.81      0.79      0.80      1053
   Интернет и СМИ       0.85      0.82      0.83      1509
             Крым       0.75      0.76      0.76       133
    Культпросвет        0.52      0.59      0.55        66
         Культура       0.87      0.91      0.89      1445
          Легпром       1.00      0.20      0.33         5
              Мир       0.87      0.88      0.87      3192
  Наука и техника       0.87      0.91      0.89      1221
      Путешествия       0.83      0.85      0.84       723
           Россия       0.83      0.80      0.82      3349
Силовые структуры       0.78      0.80      0.79      1603
           

word2vec-эмбеддинги

In [60]:
#!pip install gensim
import gensim
from gensim.models import Word2Vec, KeyedVectors
from gensim.utils import simple_preprocess
import nltk
import warnings
warnings.filterwarnings('ignore')

In [61]:
def preprocess_for_w2v(text):
    """Очистка для word2vec (без стемминга, возвращает список токенов)"""
    if not text or not isinstance(text, str):
        return []

    text = html.unescape(text)
    text = text.replace('\xa0', ' ')
    text = BeautifulSoup(text, 'html.parser').get_text()
    text = text.lower()
    text = re.sub(r'[^а-яё\s]', ' ', text)

    words = text.split()
    words = [w for w in words if w not in all_stopwords and len(w) > 2]

    return words

In [63]:
# Применяем предобработку, тут я тупанул, и надо было заранее токены создавать, так как у меня выше только со стеммингом остались данные((
processed_data = []
tokenized_sentences = []

for item in raw_data:
    full_text = f"{item['title']} {item['text']}"

    clean_text = preprocess_text(full_text)
    tokens = preprocess_for_w2v(full_text)

    topic = item['topic']
    if topic is None or topic == '':
        continue

    if clean_text and len(clean_text.split()) > 10 and len(tokens) > 5:
        processed_data.append({
            'text': clean_text,
            'tokens': tokens,
            'topic': topic
        })
        tokenized_sentences.append(tokens)

In [66]:
# Фильтруем редкие темы
topic_counts = Counter([d['topic'] for d in processed_data])
min_samples_per_topic = 5
valid_topics = {topic for topic, count in topic_counts.items() if count >= min_samples_per_topic}
filtered_data = [d for d in processed_data if d['topic'] in valid_topics]

# Подготовка массивов
X_texts = [d['text'] for d in filtered_data]
X_tokens = [d['tokens'] for d in filtered_data]
y = [d['topic'] for d in filtered_data]

In [67]:
X_train_texts, X_temp_texts, y_train, y_temp = train_test_split(
    X_texts, y, test_size=0.4, random_state=42, stratify=y
)
X_val_texts, X_test_texts, y_val, y_test = train_test_split(
    X_temp_texts, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Для word2vec нужно разделить токенизированные данные
X_train_tokens, X_temp_tokens, _, _ = train_test_split(
    X_tokens, y, test_size=0.4, random_state=42, stratify=y
)
X_val_tokens, X_test_tokens, _, _ = train_test_split(
    X_temp_tokens, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

ОБУЧЕНИЕ WORD2VEC ЭМБЕДДИНГОВ

In [68]:
w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=300,      # размерность вектора
    window=5,             # контекстное окно. Меньшее окно (2-3) лучше для синтаксических отношений, большее (5-10) - для семантических.
    min_count=5,          # минимальная частота слова. Слова реже 5 вхождений удаляются - они не несут полезной информации.
    sg=1,                 # 1 = Skip-gram, 0 = CBOW. Для новостного корпуса с разнообразной лексикой SG предпочтительнее
    negative=5,           # количество негативных семплов
    hs=0,                 # отключаем hierarchical softmax. Быстрее
    alpha=0.025,          # начальный learning rate
    min_alpha=0.0001,     # конечный learning rate
    epochs=10,            # количество эпох
    workers=4,            # количество потоков
    seed=42               # для воспроизводимости
)

print(f"Размер словаря: {len(w2v_model.wv)} слов")
print(f"Размерность векторов: {w2v_model.wv.vector_size}")


Размер словаря: 100275 слов
Размерность векторов: 300


ВИЗУАЛЬНАЯ ОЦЕНКА КАЧЕСТВА ЭМБЕДДИНГОВ

In [73]:
# 1. Самые частотные слова
all_words = []
for tokens in X_train_tokens:
    all_words.extend(tokens)

word_freq = Counter(all_words)
print("\nТоп-20 самых частотных слов:")
for word, freq in word_freq.most_common(5):
    print(f"  {word}: {freq}")

# 2. Слова, характерные для конкретных тем
print("\nАнализ ключевых слов по темам (можно определить вручную):")
unique_topics = set(y_train)
for topic in list(unique_topics)[:2]:
    topic_docs = [X_train_tokens[i] for i, t in enumerate(y_train) if t == topic]
    topic_words = [w for doc in topic_docs for w in doc]
    topic_freq = Counter(topic_words).most_common(5)
    print(f"\nТема {topic}:")
    for word, freq in topic_freq[:5]:
        print(f"    {word}: {freq}")


Топ-20 самых частотных слов:
  россии: 44733
  сообщает: 25349
  году: 24416
  сша: 23843
  время: 21353

Анализ ключевых слов по темам (можно определить вручную):

Тема 69-я параллель:
    россии: 266
    году: 257
    рублей: 234
    тасс: 206
    сообщает: 201

Тема Культпросвет :
    россии: 270
    культуры: 215
    мединский: 178
    министр: 114
    владимир: 105


In [75]:
#most_similar - поиск семантически близких слов:")
test_words = ['россия', 'сообщает', 'году', 'сша', 'время']

for word in test_words:
    similar = w2v_model.wv.most_similar(word, topn=5)
    print(f"\nСлова, похожие на '{word}':")
    for w, score in similar:
        print(f"    {w}: {score:.4f}")


#doesnt_match - поиск лишнего слова:")
test_groups = [
    ['россии', 'году', 'рублей', 'тасс', 'сообщает'],
    ['россии', 'культуры', 'мединский', 'министр', 'владимир']
]

for group in test_groups:
    valid_words = [w for w in group if w in w2v_model.wv]
    if len(valid_words) >= 3:
        odd = w2v_model.wv.doesnt_match(valid_words)
        print(f"  {valid_words} -> лишнее: '{odd}'")



Слова, похожие на 'россия':
    белоруссия: 0.4910
    страна: 0.4826
    украина: 0.4763
    турция: 0.4639
    молдавия: 0.4430

Слова, похожие на 'сообщает':
    пишет: 0.5791
    ссылкой: 0.5635
    новини: 0.5016
    финское: 0.4986
    нськ: 0.4757

Слова, похожие на 'году':
    прошлом: 0.5238
    годах: 0.5033
    текущем: 0.4821
    кельвином: 0.4695
    сельхозгоду: 0.4610

Слова, похожие на 'сша':
    соединенных: 0.6150
    штатов: 0.5691
    американские: 0.5333
    штаты: 0.5225
    соединенные: 0.5224

Слова, похожие на 'время':
    настоящее: 0.6772
    ближайшее: 0.4885
    долгое: 0.4469
    эберштейном: 0.4102
    некоторое: 0.4031
  ['россии', 'году', 'рублей', 'тасс', 'сообщает'] -> лишнее: 'рублей'
  ['россии', 'культуры', 'мединский', 'министр', 'владимир'] -> лишнее: 'россии'


In [ ]:
#Загрузите предобученные эмбеддинги из navec или rusvectores (на ваш вкус)

In [69]:
import urllib.request
import gensim
urllib.request.urlretrieve(
    "https://rusvectores.org/static/models/rusvectores4/ruwikiruscorpora/ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz",
    "ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz"
)

model_path = 'ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz'
model_ru = gensim.models.KeyedVectors.load_word2vec_format(model_path)

In [ ]:
#Дальше я не успел